# Mini Project: A Content-Based Movie Recommender (MovieLens)

## 1. Load the data

In [5]:
import os, io, re, zipfile, urllib.request
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = "ml-latest-small"
URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

def make_synthetic(seed=0):
    """Small fake dataset with the same schema as MovieLens (for offline testing only)."""
    rng = np.random.default_rng(seed)
    G = ["Action","Adventure","Animation","Comedy","Crime","Drama",
         "Fantasy","Horror","Romance","Sci-Fi","Thriller"]
    n_movies, n_users = 400, 120
    mg = ["|".join(rng.choice(G, size=rng.integers(1, 4), replace=False)) for _ in range(n_movies)]
    movies = pd.DataFrame({
        "movieId": np.arange(1, n_movies + 1),
        "title": [f"Movie {i} ({rng.integers(1980, 2019)})" for i in range(1, n_movies + 1)],
        "genres": mg})
    pop = rng.zipf(1.5, n_movies).astype(float); pop /= pop.sum()
    rows = []
    for u in range(1, n_users + 1):
        pref = {g: rng.normal(0, 1) for g in G}
        for m in rng.choice(n_movies, size=rng.integers(25, 80), replace=False, p=pop):
            s = np.mean([pref[g] for g in mg[m].split("|")])
            r = float(np.clip(round((3.5 + s + rng.normal(0, 0.5)) * 2) / 2, 0.5, 5))
            rows.append((u, m + 1, r, 1_300_000_000 + int(rng.integers(0, 10 * 365 * 86400))))
    ratings = pd.DataFrame(rows, columns=["userId", "movieId", "rating", "timestamp"])
    tags = pd.DataFrame(columns=["userId", "movieId", "tag", "timestamp"])
    return movies, ratings, tags

def load_movielens():
    try:
        if not os.path.exists(DATA_DIR):
            blob = urllib.request.urlopen(URL, timeout=30).read()
            zipfile.ZipFile(io.BytesIO(blob)).extractall(".")
        movies = pd.read_csv(f"{DATA_DIR}/movies.csv")
        ratings = pd.read_csv(f"{DATA_DIR}/ratings.csv")
        tags = pd.read_csv(f"{DATA_DIR}/tags.csv")
        return movies, ratings, tags, True
    except Exception as e:
        print("WARNING: could not load MovieLens (%s). Using SYNTHETIC data." % type(e).__name__)
        return (*make_synthetic(), False)

movies, ratings, tags, REAL = load_movielens()
print("real MovieLens:", REAL)
print(len(movies), "movies |", ratings.userId.nunique(), "users |", len(ratings), "ratings")
movies.head()

real MovieLens: True
9742 movies | 610 users | 100836 ratings


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [6]:
# Extracting movies year and for missing one filling it with median year

movies["year"] = movies.title.str.extract(r"\((\d{4})\)\s*$")[0].astype(float)
movies["year"] = movies["year"].fillna(movies["year"].median())
movies

,movieId,title,genres,year
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0
4,5,Father of the Bride Part II (1995),Comedy,1995.0
...,...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy,2017.0
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy,2017.0
9739,193585,Flint (2017),Drama,2017.0
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation,2018.0


### Converting movie titles into feature vectors based on generes using One Hot encodings

In [17]:
onehot =  movies.genres.str.get_dummies(sep="|").drop(columns=["(no genres listed)"], errors="ignore")
onehot

,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9737,1,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
9738,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
9739,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
9740,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [18]:
H = onehot.values.astype(float)
H

array([[0., 1., 1., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(9742, 19))

In [19]:
onehot.columns

Index(['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime',
       'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX',
       'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War',
       'Western'],
      dtype='str')

### Converting movie titles into feature vectors based on "generes" and "tags" using TF-IDF

In [51]:
tag_text = (
    tags.assign(tag=tags["tag"].astype(str).str.lower())
    .groupby("movieId")["tag"]
    .apply("|".join)
)
tag_text.head(10)
# .assign does the preprocessing steps of converting tags to string and then to lower {Important in language processing}

movieId
1                                  pixar|pixar|fun
2     fantasy|magic board game|robin williams|game
3                                        moldy|old
5                                 pregnancy|remake
7                                           remake
11                              politics|president
14                              politics|president
16                                           mafia
17                                     jane austen
21                                       hollywood
Name: tag, dtype: str

In [53]:
# Bringing tag_text at movieId level from tags dataset
movies["tag_text"] = movies.movieId.map(tag_text).fillna("")   # it's like a V-lookup on movieId column
movies.head()

,movieId,title,genres,year,tag_text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,pixar|pixar|fun
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,fantasy|magic board game|robin williams|game
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,moldy|old
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,
4,5,Father of the Bride Part II (1995),Comedy,1995.0,pregnancy|remake


#### Why can't One-Hot be applied on combination of "generes" and "tag_text"?

In [72]:
# Text pre-processing for generes and tag_text for creating "doc". It should be done for making TF-IDF efficient

movies["doc"] = movies.genres.fillna("").str.lower().str.strip()+"|"+movies.tag_text.fillna("").str.lower().str.strip()
movies.head()

,movieId,title,genres,year,tag_text,doc
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,pixar|pixar|fun,adventure|animation|children|comedy|fantasy|pi...
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,fantasy|magic board game|robin williams|game,adventure|children|fantasy|fantasy|magic board...
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,moldy|old,comedy|romance|moldy|old
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,,comedy|drama|romance|
4,5,Father of the Bride Part II (1995),Comedy,1995.0,pregnancy|remake,comedy|pregnancy|remake


### TF-IDF tokenizer

In [74]:
tfidf = TfidfVectorizer(
    token_pattern=r"[A-Za-z0-9\-']+",
    lowercase=True,
    stop_words="english"
)
X = tfidf.fit_transform(movies["doc"])
print("TF-IDF shape:", X.shape)

TF-IDF shape: (9742, 1696)


## Implementing different functions to calculate different similarity metric between movies

In [85]:
# A function that finds the movie based on keywords and returns its index/

def find_movie(query):
    hit = movies[movies.title.str.contains(query, case=False, regex=False)]
    if hit.empty:
        raise ValueError(f"no movie matching {query!r}")
    return hit.index[0]

i = find_movie("Waiting")
print(f"The Index: {i} is matching the movie string match from the list below")

The Index: 3 is matching the movie string match from the list below


In [83]:
movies.head()

,movieId,title,genres,year,tag_text,doc
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,pixar|pixar|fun,adventure|animation|children|comedy|fantasy|pi...
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,fantasy|magic board game|robin williams|game,adventure|children|fantasy|fantasy|magic board...
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,moldy|old,comedy|romance|moldy|old
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,,comedy|drama|romance|
4,5,Father of the Bride Part II (1995),Comedy,1995.0,pregnancy|remake,comedy|pregnancy|remake


### Check which similarity metric to use when(Cosine, Eucledian, Jaccard) and with One-Hot or TF-IDF which similarity makes sense

In [87]:
# Implementig a function to calculate cosine similarity for TF-IDF movie feature

In [91]:
X.shape
print(f"{X.shape}:{X[i]}")

(9742, 1696):<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3 stored elements and shape (1, 1696)>
  Coords	Values
  (0, 322)	0.5048999073185984
  (0, 1286)	0.7261829999003466
  (0, 458)	0.4666201177032598


In [100]:
cos_t = cosine_similarity(X, X[i]).ravel()
cos_t

array([0.04835315, 0.        , 0.21827351, ..., 0.46662012, 0.        ,
       0.50489991], shape=(9742,))

In [101]:
order = np.argsort(-cos_t)
order

array([7818, 9299, 6481, ..., 8009, 1525, 4690], shape=(9742,))

In [113]:
movies.iloc[order][['title','genres']]

,title,genres
7818,"For a Good Time, Call... (2012)",Comedy|Drama|Romance
9299,All Yours (2016),Comedy|Drama|Romance
6481,Knocked Up (2007),Comedy|Drama|Romance
8471,And So It Goes (2014),Comedy|Drama|Romance
2341,Shampoo (1975),Comedy|Drama|Romance
...,...,...
4118,Equilibrium (2002),Action|Sci-Fi|Thriller
1472,"Exorcist, The (1973)",Horror|Mystery
8009,[REC]³ 3 Génesis (2012),Horror|Thriller
1525,"Incredible Journey, The (1963)",Adventure|Children


In [122]:
# Finding a movie -> Finding it's similarity to all otehr movie -> Selecting top-n similar movie{excluding itself} and returning

def top_cosine_similar(query, n=5):
    i = find_movie(query)
    cos_t = cosine_similarity(X, X[i]).ravel()
    order = np.argsort(-cos_t)
    order = [j for j in order if j != i][:n]
    return movies.iloc[order][['title','genres']][:n]
top_cosine_similar("toy", n=5)

,title,genres
1757,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
8695,Guardians of the Galaxy 2 (2017),Action|Adventure|Sci-Fi
6948,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy
9430,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy


## Preparing Training and Testing Data set for user Profiling

### In here at user level 80% of the data should go into training set and 20% should go into testing set as we are creating recommendations at user level

In [147]:
def split_per_user(ratings, test_frac=0.2, min_ratings=10):
    train_parts, test_parts = [], []

    for u, g in ratings.groupby("userId"):
        if len(g) < min_ratings: # if the current user had rated less than 10 movies then skip this user
            continue

        g = g.sample(             # Shuffel
            frac=1.0,
            random_state=42
        )

        n_test = max(1, int(round(len(g) * test_frac)))          # No. of testing examples

        test_parts.append(g.iloc[:n_test])                       # Selecting from shuffeled
        train_parts.append(g.iloc[n_test:])                      # have a careful look at the function of iloc in here.

    return pd.concat(train_parts), pd.concat(test_parts)

train_parts, test_parts = split_per_user(ratings, test_frac=0.2, min_ratings=10)

In [144]:
ratings

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


### Datasets: movies, ratings, tags; &nbsp; &nbsp;&nbsp; &nbsp; find_movie("toy") --> returns the index from "movies"
### H: One-Hot; &nbsp; &nbsp;&nbsp; &nbsp; X: TF-IDF;      